In [1]:
#  Load Cleaned Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: "%.4f" % x)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

heart = pd.read_csv("../../data/health_risk/heart_cleaned.csv")
diabetes = pd.read_csv("../../data/health_risk/diabetes_cleaned.csv")

print("Heart shape    :", heart.shape)
print("Diabetes shape :", diabetes.shape)

Heart shape    : (1025, 14)
Diabetes shape : (768, 9)


In [2]:
# Heart Disease Feature Engineering
print("HEART DISEASE FEATURE ENGINEERING")
print()

h = heart.copy()

h["age_group"] = pd.cut(h["age"], bins=[0,40,50,60,70,100],
                         labels=["<40","40-50","50-60","60-70","70+"])
h["age_encoded"] = pd.cut(h["age"], bins=[0,40,50,60,70,100], labels=[1,2,3,4,5]).astype(float)
h["high_bp"] = (h["trestbps"] > 140).astype(int)
h["high_chol"] = (h["chol"] > 240).astype(int)
h["low_max_hr"] = (h["thalach"] < 120).astype(int)
h["hr_reserve"] = 220 - h["age"] - h["thalach"]
h["hr_reserve_pct"] = h["hr_reserve"] / (220 - h["age"])
h["chest_pain_severity"] = h["cp"].map({0: 0, 1: 3, 2: 2, 3: 1})
h["cardiac_risk_score"] = (
    h["age_encoded"] / 5 * 0.20 +
    (h["trestbps"] / 200) * 0.15 +
    (h["chol"] / 600) * 0.10 +
    h["exang"] * 0.20 +
    (h["oldpeak"] / 6.2) * 0.20 +
    (h["ca"] / 4) * 0.15
)
h["has_exercise_angina"] = h["exang"]
h["multiple_risk_factors"] = (
    (h["high_bp"].astype(int) + h["high_chol"].astype(int) +
     h["fbs"].astype(int) + h["exang"].astype(int)) >= 2
).astype(int)
h = h.drop(columns=["age_group"])

print("Heart features created:")
new_feats = ["age_encoded", "high_bp", "high_chol", "low_max_hr", "hr_reserve",
             "hr_reserve_pct", "chest_pain_severity", "cardiac_risk_score",
             "has_exercise_angina", "multiple_risk_factors"]
for f in new_feats:
    d_mean = h[h["target"]==1][f].mean()
    nd_mean = h[h["target"]==0][f].mean()
    print(f"  {f:<28}: disease={d_mean:.4f}  no_disease={nd_mean:.4f}")

h.to_csv("../../data/health_risk/heart_engineered.csv", index=False)
print()
print(f"Heart engineered saved | shape: {h.shape}")

HEART DISEASE FEATURE ENGINEERING

Heart features created:
  age_encoded                 : disease=2.7567  no_disease=3.0741
  high_bp                     : disease=0.1616  no_disease=0.2645
  high_chol                   : disease=0.4221  no_disease=0.5631
  low_max_hr                  : disease=0.0399  no_disease=0.1964
  hr_reserve                  : disease=9.0057  no_disease=24.3006
  hr_reserve_pct              : disease=0.0541  no_disease=0.1476
  chest_pain_severity         : disease=1.6939  no_disease=0.5110
  cardiac_risk_score          : disease=0.3066  no_disease=0.4703
  has_exercise_angina         : disease=0.1350  no_disease=0.5491
  multiple_risk_factors       : disease=0.1920  no_disease=0.4790

Heart engineered saved | shape: (1025, 24)


In [3]:
# Diabetes Feature Engineering
print("DIABETES FEATURE ENGINEERING")
print()

d = diabetes.copy()

d["high_glucose"] = (d["Glucose"] > 140).astype(int)
d["obese"] = (d["BMI"] >= 30).astype(int)
d["high_age"] = (d["Age"] >= 45).astype(int)
d["high_pregnancies"] = (d["Pregnancies"] >= 4).astype(int)
d["glucose_bmi_interaction"] = d["Glucose"] * d["BMI"] / 1000
d["insulin_resistance"] = d["Glucose"] / (d["Insulin"] + 1)
d["bmi_age_risk"] = d["BMI"] * d["Age"] / 1000
d["glucose_normalized"] = (d["Glucose"] - d["Glucose"].min()) / (d["Glucose"].max() - d["Glucose"].min())
d["bmi_normalized"] = (d["BMI"] - d["BMI"].min()) / (d["BMI"].max() - d["BMI"].min())
d["age_normalized"] = (d["Age"] - d["Age"].min()) / (d["Age"].max() - d["Age"].min())
d["diabetes_risk_score"] = (
    d["glucose_normalized"] * 0.40 +
    d["bmi_normalized"] * 0.25 +
    d["age_normalized"] * 0.15 +
    (d["Pregnancies"] / d["Pregnancies"].max()) * 0.10 +
    d["DiabetesPedigreeFunction"] / d["DiabetesPedigreeFunction"].max() * 0.10
)
d["multiple_risk_factors"] = (
    (d["high_glucose"].astype(int) + d["obese"].astype(int) +
     d["high_age"].astype(int) + d["high_pregnancies"].astype(int)) >= 2
).astype(int)

print("Diabetes features created:")
new_feats_d = ["high_glucose", "obese", "high_age", "glucose_bmi_interaction",
               "insulin_resistance", "diabetes_risk_score", "multiple_risk_factors"]
for f in new_feats_d:
    pos_mean = d[d["Outcome"]==1][f].mean()
    neg_mean = d[d["Outcome"]==0][f].mean()
    print(f"  {f:<28}: diabetic={pos_mean:.4f}  non-diabetic={neg_mean:.4f}")

d.to_csv("../../data/health_risk/diabetes_engineered.csv", index=False)
print()
print(f"Diabetes engineered saved | shape: {d.shape}")

DIABETES FEATURE ENGINEERING

Diabetes features created:
  high_glucose                : diabetic=0.4925  non-diabetic=0.1200
  obese                       : diabetic=0.8246  non-diabetic=0.5240
  high_age                    : diabetic=0.2463  non-diabetic=0.1340
  glucose_bmi_interaction     : diabetic=5.0396  non-diabetic=3.4376
  insulin_resistance          : diabetic=1.0500  non-diabetic=1.0568
  diabetes_risk_score         : diabetic=0.4326  non-diabetic=0.2996
  multiple_risk_factors       : diabetic=0.7239  non-diabetic=0.3280

Diabetes engineered saved | shape: (768, 21)


In [4]:
# Feature Importance
from sklearn.ensemble import RandomForestClassifier

print("FEATURE IMPORTANCE — HEART DISEASE")
print()

h = pd.read_csv("../../data/health_risk/heart_engineered.csv")
X_h = h.drop(columns=["target"])
y_h = h["target"]
rf_h = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_h.fit(X_h, y_h)
imp_h = pd.Series(rf_h.feature_importances_, index=X_h.columns).sort_values(ascending=False)
print("Top 10 features:")
for i, (feat, val) in enumerate(imp_h.head(10).items(), 1):
    tag = "ENGINEERED" if feat not in heart.columns else "original"
    print(f"  {i:2}. {feat:<28} {val:.6f}  [{tag}]")

print()
print("FEATURE IMPORTANCE — DIABETES")
print()

d = pd.read_csv("../../data/health_risk/diabetes_engineered.csv")
X_d = d.drop(columns=["Outcome"])
y_d = d["Outcome"]
rf_d = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_d.fit(X_d, y_d)
imp_d = pd.Series(rf_d.feature_importances_, index=X_d.columns).sort_values(ascending=False)
print("Top 10 features:")
for i, (feat, val) in enumerate(imp_d.head(10).items(), 1):
    tag = "ENGINEERED" if feat not in diabetes.columns else "original"
    print(f"  {i:2}. {feat:<28} {val:.6f}  [{tag}]")

eng_h = imp_h[[f for f in imp_h.index if f not in heart.columns]].sum()
orig_h = imp_h[[f for f in imp_h.index if f in heart.columns]].sum()
eng_d = imp_d[[f for f in imp_d.index if f not in diabetes.columns]].sum()
orig_d = imp_d[[f for f in imp_d.index if f in diabetes.columns]].sum()

print()
print(f"Heart   — Engineered features: {eng_h/(eng_h+orig_h)*100:.1f}% importance")
print(f"Diabetes — Engineered features: {eng_d/(eng_d+orig_d)*100:.1f}% importance")
print()
print("Notebook 2 complete.")

FEATURE IMPORTANCE — HEART DISEASE

Top 10 features:
   1. cardiac_risk_score           0.158998  [ENGINEERED]
   2. ca                           0.085789  [original]
   3. thal                         0.077457  [original]
   4. cp                           0.076605  [original]
   5. chest_pain_severity          0.070917  [ENGINEERED]
   6. oldpeak                      0.065633  [original]
   7. hr_reserve_pct               0.061657  [ENGINEERED]
   8. thalach                      0.061467  [original]
   9. hr_reserve                   0.052976  [ENGINEERED]
  10. age                          0.049825  [original]

FEATURE IMPORTANCE — DIABETES

Top 10 features:
   1. diabetes_risk_score          0.133621  [ENGINEERED]
   2. glucose_bmi_interaction      0.108930  [ENGINEERED]
   3. glucose_normalized           0.085117  [ENGINEERED]
   4. Glucose                      0.081252  [original]
   5. bmi_age_risk                 0.074641  [ENGINEERED]
   6. DiabetesPedigreeFunction     0.06459